## API Call

In [1]:

import re
import unicodedata
import requests
import pandas as pd


def get_nih_awards(full_name, rpp=25, name_aliases=None):

    url = "https://api.reporter.nih.gov/v2/projects/search"

    def clean(value):
        if pd.isna(value):
            return ""
        value = str(value).strip()
        return "" if value.lower() == "nan" else value

    def normalize(value):
        value = unicodedata.normalize("NFKD", clean(value))
        value = "".join(
            c for c in value if not unicodedata.combining(c)
        )
        return re.sub(r"[^a-z0-9]", "", value.casefold())

    full_name = clean(full_name)
    name_parts = full_name.split()

    if len(name_parts) < 2:
        raise ValueError("full_name must contain first and last names.")

    first, last = name_parts[0], name_parts[-1]
    middle = " ".join(name_parts[1:-1])

    if not isinstance(rpp, int) or isinstance(rpp, bool) or rpp < 1:
        raise ValueError("rpp must be a positive integer.")

    limit = min(rpp, 500)

    search_names = [
        {"first_name": first, "last_name": last}
    ]

    if middle:
        search_names.append({
            "first_name": first,
            "middle_name": middle,
            "last_name": last
        })

    search_names.extend(
        {"first_name": clean(alias), "last_name": last}
        for alias in (name_aliases or [])
        if clean(alias)
    )

    unique_names = list({
        tuple(sorted(name.items())): name
        for name in search_names
    }.values())

    projects_by_id = {}

    with requests.Session() as session:

        for search_name in unique_names:
            offset = 0

            while True:

                response = session.post(
                    url,
                    json={
                        "criteria": {
                            "pi_names": [search_name],
                            "include_active_projects": True
                        },
                        "offset": offset,
                        "limit": limit
                    },
                    timeout=30
                )

                response.raise_for_status()
                projects = response.json().get("results") or []

                for project in projects:
                    project_id = (
                        project.get("appl_id")
                        or project.get("project_num")
                        or project.get("project_title")
                    )

                    if project_id is not None:
                        projects_by_id[str(project_id)] = project

                if len(projects) < limit:
                    break

                offset += limit

                if offset > 14999:
                    print("Warning: NIH API offset limit reached.")
                    break

    target_first = normalize(first)
    target_middle = normalize(middle)
    target_last = normalize(last)

    first_aliases = {
        target_first,
        *[normalize(alias) for alias in (name_aliases or []) if normalize(alias)]
    }

    def first_name_matches(candidate):
        candidate = normalize(candidate)

        return bool(candidate) and any(
            candidate == alias
            or (len(candidate) == 1 and alias.startswith(candidate))
            or (len(alias) == 1 and candidate.startswith(alias))
            for alias in first_aliases
        )

    def investigator_name(person):
        return " ".join(
            clean(person.get(key))
            for key in ("first_name", "middle_name", "last_name")
            if clean(person.get(key))
        )

    records = []

    for project in projects_by_id.values():

        investigators = project.get("principal_investigators") or []

        matched = [
            person for person in investigators
            if normalize(person.get("last_name")) == target_last
            and first_name_matches(person.get("first_name"))
        ]

        if len(matched) != 1:
            continue

        person = matched[0]
        contact_status = person.get("is_contact_pi")

        if not isinstance(contact_status, bool):
            continue

        person_middle = normalize(person.get("middle_name"))

        middle_status = (
            "Not supplied"
            if not target_middle or not person_middle
            else "Match"
            if target_middle == person_middle
            else "Initial-compatible"
            if target_middle[0] == person_middle[0]
            else "Different"
        )

        records.append({
            "Matched Name": investigator_name(person),
            "Role": "PI" if contact_status else "Co-PI",
            "Participants": "; ".join(
                filter(None, map(investigator_name, investigators))
            ),
            "Title": project.get("project_title"),
            "Abstract": project.get("abstract_text"),
            "Award Type": project.get("activity_code"),
            "Journal": pd.NA,
            "Amount": project.get("award_amount"),
            "NIH Email": pd.NA,
            "Institute": (
                (project.get("organization") or {}).get("org_name")
                or project.get("organization_name")
                or pd.NA
            ),
            "Start Date": project.get("project_start_date"),
            "End Date": project.get("project_end_date")
        })

    df = pd.DataFrame(records)

    df["Full Name"] = full_name

    output_columns = [
        "Full Name", "Participants", "Searched Email",
        "Matched Name", "Role", "Title", "Abstract",
        "Award Type", "Journal", "Amount", "NIH Email",
        "Institute", "Start Date", "End Date"
    ]

    return df.reindex(columns=output_columns)


In [2]:
df = pd.read_excel("mhti_accepted.xlsx")[["name"]]

print(df)

                 name
0       Alicia Bargar
1    Andrew D. Miller
2      Audrey Boruvka
3             Bin Xie
4            Dan Ding
..                ...
338     Natasha Chaku
339       Phoebe Tran
340    Tyler Prochnow
341       Xueqing Liu
342        Yiwen Dong

[343 rows x 1 columns]


In [ ]:
import time
import requests
import pandas as pd


def get_nih_awards_for_people(
    people,
    delay=0.2,
    rpp=25,
    name_aliases_by_email=None
):

    all_results = []

    for _, person in people.iterrows():

        full_name = person["name"]

        print(f"  Searching {full_name}")
        
        try:
        
            result = get_nih_awards(
                full_name=full_name
            )

            if not result.empty:
                all_results.append(result)

                print(
                    f"  Retrieved {len(result)} matching project(s)"
                )

                # Report records requiring identity or role review.
                if "Match Status" in result.columns:
                    needs_review = result["Match Status"].isin([
                        "No investigator match",
                        "Multiple investigator matches",
                        "Role unverified"
                    ]).sum()

                    if needs_review:
                        print(
                            f"  Warning: {needs_review} project(s) "
                            "need investigator/role verification"
                        )
            else:
                print("  No matching projects found")

        except requests.exceptions.RequestException as e:
            print(
                f"  API error for {full_name}: {e}"
            )

        except Exception as e:
            print(
                f"  Error for {full_name}: {e}"
            )

        time.sleep(delay)

    if not all_results:
        return pd.DataFrame(columns=[
            "Full Name",
            "Searched Email",
            "Match Name",
            "Role",
            "Participants",
            "Title",
            "Abstract",
            "Award Type",
            "Journal",
            "Amount",
            "NIH Email",
            "Institute",
            "Start Date",
            "End Date"
        ])

    return pd.concat(all_results, ignore_index=True)

In [4]:
results = get_nih_awards_for_people(df)

  No matching projects found
  Retrieved 4 matching project(s)
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  Retrieved 1 matching project(s)
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  Retrieved 1 matching project(s)
  No matching projects found
  Retrieved 3 matching project(s)
  Retrieved 2 matching project(s)
  Retrieved 4 matching project(s)
  Retrieved 1 matching project(s)
  Retrieved 3 matching project(s)
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  No matching projects found
  Retrieved 3 matching project(s

In [5]:
results["Role"].unique()

array(['Co-PI', 'PI'], dtype=object)

In [6]:
results

,Full Name,Participants,Searched Email,Matched Name,Role,Title,Abstract,Award Type,Journal,Amount,NIH Email,Institute,Start Date,End Date
0,Andrew D. Miller,Jennifer C Felger; Christina Gavegnano; ANDREW...,NaN,ANDREW H MILLER,Co-PI,JAK Signaling as a Mechanism of Inflammation-r...,Project Summary\nThis project will test the hy...,R01,<NA>,627566,<NA>,EMORY UNIVERSITY,2025-05-01T00:00:00,2030-02-28T00:00:00
1,Andrew D. Miller,Jennifer C Felger; ANDREW H MILLER,NaN,ANDREW H MILLER,Co-PI,The Role of Inflammation in CNS Mechanisms of ...,PROJECT SUMMARY\nRisk of depression is substan...,R01,<NA>,618864,<NA>,EMORY UNIVERSITY,2021-09-10T00:00:00,2027-08-31T00:00:00
2,Andrew D. Miller,David Ryan Goldsmith; ANDREW H MILLER; Michael...,NaN,ANDREW H MILLER,PI,Emory Psychiatry Clinical Scientist Training P...,Project Summary\nThe Emory Psychiatry Clinical...,R25,<NA>,199392,<NA>,EMORY UNIVERSITY,2014-07-18T00:00:00,2027-01-31T00:00:00
3,Andrew D. Miller,Andrew Howard Miller,NaN,Andrew Howard Miller,PI,Light adaptation in regionally and functionall...,"PROJECT SUMMARY\nVisual tasks, like reading pr...",F32,<NA>,83224,<NA>,UNIVERSITY OF WISCONSIN-MADISON,2025-12-01T00:00:00,2027-11-30T00:00:00
4,Ju Gao,Ju Gao,NaN,Ju Gao,PI,TMEM106B and Mitochondrial Dysfunction in FTLD...,"Dementia affects millions worldwide, yet remai...",R03,<NA>,310083,<NA>,UNIVERSITY OF ARIZONA,2025-09-01T00:00:00,2027-08-31T00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,Tiffany Glynn,Tiffany Rose Glynn,NaN,Tiffany Rose Glynn,PI,Leveraging mHealth to develop syndemic-based b...,Overview. The goal of this project is to devel...,K23,<NA>,199800,<NA>,BRIGHAM AND WOMEN'S HOSPITAL,2024-06-01T00:00:00,2029-05-31T00:00:00
266,Amrik Singh Khalsa,Amrik Singh Khalsa,NaN,Amrik Singh Khalsa,PI,The Development and Evaluation of Visual Commu...,Modified Project Summary/Abstract Section\nDes...,K23,<NA>,157920,<NA>,RESEARCH INST NATIONWIDE CHILDREN'S HOSP,2022-06-01T00:00:00,2027-05-31T00:00:00
267,Darina Petrovsky,Darina V. Petrovsky; Katherine Ramos,NaN,Darina V. Petrovsky,PI,The feasibility of a dyadic music-based interv...,PROJECT SUMMARY\nPersons living with dementia ...,R34,<NA>,754231,<NA>,DUKE UNIVERSITY,2026-08-06T00:00:00,2029-07-31T00:00:00
268,Karly Ingram,Karly Mary Ingram,NaN,Karly Mary Ingram,PI,iManageAYA: Preparation and Optimization of a ...,"There are ~680,000 adolescent and young adult ...",R00,<NA>,248649,<NA>,EAST CAROLINA UNIVERSITY,2022-06-01T00:00:00,2027-08-31T00:00:00


In [8]:
results.to_excel("NIH_raw2.xlsx", index=False)

## Merge with Demographic Data

In [10]:
import pandas as pd

df_nsf = pd.read_excel("NIH_raw2.xlsx")
df_demo = pd.read_excel("mhti_accepted.xlsx")
df_demo.rename(columns={'name': 'Full Name'}, inplace=True)

In [11]:
df_nsf2 = df_nsf.copy()
df_demo2 = df_demo.copy()

# --- Merge onto NSF data ---

df_nsf2 = df_nsf2.merge(
    df_demo,
    on=["Full Name"],
    how="left"
)

df_nsf2

,Full Name,Participants,Searched Email,Matched Name,Role,Title,Abstract,Award Type,Journal,Amount,NIH Email,Institute,Start Date,End Date,cohort,affiliation,openalex_author_id,google_scholar_url,google_scholar_user_id,note
0,Andrew D. Miller,Jennifer C Felger; Christina Gavegnano; ANDREW...,NaN,ANDREW H MILLER,Co-PI,JAK Signaling as a Mechanism of Inflammation-r...,Project Summary\nThis project will test the hy...,R01,NaN,627566.0,NaN,EMORY UNIVERSITY,2025-05-01T00:00:00,2030-02-28T00:00:00,2015,University of Washington,A5027248126,https://scholar.google.com/citations?user=U5Qe...,U5QeuUcAAAAJ,NaN
1,Andrew D. Miller,Jennifer C Felger; ANDREW H MILLER,NaN,ANDREW H MILLER,Co-PI,The Role of Inflammation in CNS Mechanisms of ...,PROJECT SUMMARY\nRisk of depression is substan...,R01,NaN,618864.0,NaN,EMORY UNIVERSITY,2021-09-10T00:00:00,2027-08-31T00:00:00,2015,University of Washington,A5027248126,https://scholar.google.com/citations?user=U5Qe...,U5QeuUcAAAAJ,NaN
2,Andrew D. Miller,David Ryan Goldsmith; ANDREW H MILLER; Michael...,NaN,ANDREW H MILLER,PI,Emory Psychiatry Clinical Scientist Training P...,Project Summary\nThe Emory Psychiatry Clinical...,R25,NaN,199392.0,NaN,EMORY UNIVERSITY,2014-07-18T00:00:00,2027-01-31T00:00:00,2015,University of Washington,A5027248126,https://scholar.google.com/citations?user=U5Qe...,U5QeuUcAAAAJ,NaN
3,Andrew D. Miller,Andrew Howard Miller,NaN,Andrew Howard Miller,PI,Light adaptation in regionally and functionall...,"PROJECT SUMMARY\nVisual tasks, like reading pr...",F32,NaN,83224.0,NaN,UNIVERSITY OF WISCONSIN-MADISON,2025-12-01T00:00:00,2027-11-30T00:00:00,2015,University of Washington,A5027248126,https://scholar.google.com/citations?user=U5Qe...,U5QeuUcAAAAJ,NaN
4,Ju Gao,Ju Gao,NaN,Ju Gao,PI,TMEM106B and Mitochondrial Dysfunction in FTLD...,"Dementia affects millions worldwide, yet remai...",R03,NaN,310083.0,NaN,UNIVERSITY OF ARIZONA,2025-09-01T00:00:00,2027-08-31T00:00:00,2015,The Ohio State University,A5069150137,https://scholar.google.com/citations?user=RF0T...,RF0TKQoAAAAJ,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,Tiffany Glynn,Tiffany Rose Glynn,NaN,Tiffany Rose Glynn,PI,Leveraging mHealth to develop syndemic-based b...,Overview. The goal of this project is to devel...,K23,NaN,199800.0,NaN,BRIGHAM AND WOMEN'S HOSPITAL,2024-06-01T00:00:00,2029-05-31T00:00:00,2025,Harvard,A5000865416,https://scholar.google.com/citations?user=ph3U...,ph3UPScAAAAJ,NaN
266,Amrik Singh Khalsa,Amrik Singh Khalsa,NaN,Amrik Singh Khalsa,PI,The Development and Evaluation of Visual Commu...,Modified Project Summary/Abstract Section\nDes...,K23,NaN,157920.0,NaN,RESEARCH INST NATIONWIDE CHILDREN'S HOSP,2022-06-01T00:00:00,2027-05-31T00:00:00,2026,Nationwide Children's Hospital,A5006410885,https://scholar.google.com/citations?user=JxCx...,JxCxpeQAAAAJ,NaN
267,Darina Petrovsky,Darina V. Petrovsky; Katherine Ramos,NaN,Darina V. Petrovsky,PI,The feasibility of a dyadic music-based interv...,PROJECT SUMMARY\nPersons living with dementia ...,R34,NaN,754231.0,NaN,DUKE UNIVERSITY,2026-08-06T00:00:00,2029-07-31T00:00:00,2026,Duke University,A5062951815,https://scholar.google.com/citations?user=0_w9...,0_w93tMAAAAJ,NaN
268,Karly Ingram,Karly Mary Ingram,NaN,Karly Mary Ingram,PI,iManageAYA: Preparation and Optimization of a ...,"There are ~680,000 adolescent and young adult ...",R00,NaN,248649.0,NaN,EAST CAROLINA UNIVERSITY,2022-06-01T00:00:00,2027-08-31T00:00:00,2026,East Carolina University,A5124932165,https://scholar.google.com/citations?user=zIqJ...,zIqJE_IAAAAJ,NaN


In [12]:
df_nsf2.to_excel("NIH_demographic2.xlsx", index=False)